# Capítulo 7 — Zoneamento agroclimático via ISNA

**Curso:** Agrometeorologia Operacional com Python
**Prof. Dr. Fabrício Correia de Oliveira** — UTFPR, Campus Santa Helena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologia/blob/main/curso/07_isna_zoneamento.ipynb)

> Pré-requisito: Capítulos 1 a 6 (especialmente `amp.balanco_hidrico_climatologico`, do Capítulo 6).

---

## 7.1 Motivação

Até aqui, o balanço hídrico (Capítulo 6) usava a evapotranspiração de **referência** (`ETP`,
uma superfície gramada padrão) como demanda. Mas cada cultura tem sua própria curva de
demanda ao longo do ciclo (`ETc = Kc × ETo`, vista no Capítulo 6 da apostila). Quando
comparamos quanto água a cultura **realmente recebeu** (`ETR`) com quanto ela **precisava**
(`ETc`) ao longo do ciclo inteiro, temos o **ISNA — Índice de Satisfação das Necessidades de
Água**.

O ISNA é a base metodológica de zoneamento de risco climático para culturas anuais — é
exatamente essa lógica que sustenta o trabalho de recomendação de janela de semeadura para
milho segunda safra no oeste do Paraná. Neste capítulo vamos construir essa mesma lógica,
em versão didática, e comparar municípios e janelas de semeadura.


## 7.2 Conceito e fórmula

$$ISNA = \frac{\sum ETR}{\sum ETc}$$

em que a soma é feita sobre todos os períodos (meses ou decêndios) do ciclo da cultura, e
`ETR` (evapotranspiração real) vem do mesmo balanço hídrico do Capítulo 6 — só que agora a
demanda de entrada é `ETc`, não `ETP`.

`ISNA = 1` significa que a cultura nunca sofreu restrição hídrica no ciclo. Valores mais
baixos indicam maior risco de déficit hídrico durante o ciclo — e é isso que orienta a
escolha da melhor janela de semeadura.


## 7.3 Do papel ao código

A `agrometeorologiapy` já tem uma função pronta para isso: `amp.balanco_hidrico_cultura(df)`,
que espera um DataFrame com as colunas `'Chuva'`, `'ETc'` e `'CAD'` e devolve o balanço
período a período, já incluindo uma coluna `'ISNA'` por período. A única coisa que a
biblioteca não faz por nós é o **agregado do ciclo inteiro** (`ISNA = ΣETR / ΣETc`, e não a
média dos ISNA mensais) — por isso mantemos aqui um pequeno wrapper, `calcular_isna`, que só
soma `ETR` e `ETc` sobre o balanço devolvido pela biblioteca.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import agrometeorologiapy as amp


def calcular_isna(P: pd.Series, ETc: pd.Series, CAD: float = 100.0) -> tuple[float, pd.DataFrame]:
    """
    ISNA = soma(ETR) / soma(ETc) ao longo do ciclo da cultura, usando
    amp.balanco_hidrico_cultura (Thornthwaite & Mather) como núcleo do cálculo.
    Retorna (isna, balanco_detalhado).
    """
    df = pd.DataFrame({"Chuva": P.to_numpy(), "ETc": ETc.to_numpy(), "CAD": CAD}, index=P.index)
    balanco = amp.balanco_hidrico_cultura(df)
    isna = balanco["ETR"].sum() / balanco["ETc"].sum()
    return isna, balanco

## 7.4 Atividade guiada — janelas de semeadura em múltiplos municípios (dados didáticos)

> **Atenção:** os dados de precipitação e ETo abaixo são **ilustrativos**, criados só para
> praticar a lógica do ISNA — não são observações reais. Para um projeto de verdade, substitua
> por dados baixados via NASA POWER (Capítulo 1) ou TerraClimate/GEE, agregados por município.

Vamos simular 5 municípios representando regiões diferentes do Paraná (mesma lógica de
seleção do PBL de zoneamento), com um ciclo de milho segunda safra de 4 "meses" e curva de
`Kc` simplificada (`0,40 → 0,70 → 1,15 → 1,15`, aproximando as fases da Tabela 6.2 da
apostila), testando três janelas de semeadura: janeiro, fevereiro e março.


In [ ]:
municipios = {
    "Oeste (Cascavel)":        {"P": [180, 150, 130, 90, 80, 60],  "ETo": [150, 130, 110, 80, 60, 45]},
    "Noroeste (Paranavaí)":    {"P": [140, 110, 90, 60, 50, 40],   "ETo": [160, 140, 120, 90, 70, 50]},
    "Norte (Londrina)":        {"P": [160, 140, 120, 80, 70, 55],  "ETo": [145, 125, 105, 78, 58, 42]},
    "Centro-Sul (Guarapuava)": {"P": [150, 130, 140, 110, 100, 90],"ETo": [110, 95, 85, 65, 50, 35]},
    "Sudoeste (Palmas)":       {"P": [170, 150, 150, 120, 110, 95],"ETo": [115, 100, 88, 68, 52, 38]},
}

meses_simulados = ["M1", "M2", "M3", "M4", "M5", "M6"]
Kc_ciclo = [0.40, 0.70, 1.15, 1.15]  # curva simplificada, 4 "meses" de ciclo
janelas_semeadura = {"Janeiro": 0, "Fevereiro": 1, "Março": 2}

CAD = 100.0
resultados = []

for municipio, dados in municipios.items():
    for nome_janela, offset in janelas_semeadura.items():
        P_ciclo = pd.Series(dados["P"][offset:offset+4], index=meses_simulados[:4])
        ETo_ciclo = pd.Series(dados["ETo"][offset:offset+4], index=meses_simulados[:4])
        ETc_ciclo = ETo_ciclo * pd.Series(Kc_ciclo, index=meses_simulados[:4])

        isna, _ = calcular_isna(P_ciclo, ETc_ciclo, CAD=CAD)
        resultados.append({"municipio": municipio, "janela_semeadura": nome_janela, "ISNA": round(isna, 3)})

df_isna = pd.DataFrame(resultados)
tabela_isna = df_isna.pivot(index="municipio", columns="janela_semeadura", values="ISNA")
tabela_isna


In [ ]:
melhor_janela_por_municipio = df_isna.loc[df_isna.groupby("municipio")["ISNA"].idxmax()]
melhor_janela_por_municipio[["municipio", "janela_semeadura", "ISNA"]].reset_index(drop=True)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
tabela_isna.plot(kind="bar", ax=ax)
ax.set_ylabel("ISNA")
ax.set_ylim(0, 1.05)
ax.set_title("ISNA por município e janela de semeadura (dados ilustrativos)")
ax.legend(title="Janela de semeadura")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 7.5 Desafio — comparando com o critério do PBL do abacaxi

O PBL de zoneamento do abacaxizeiro classifica risco hídrico pela **deficiência hídrica anual
(Dha)**: baixo risco se `Dha < 100 mm`. O ISNA usa uma lógica relacionada, mas não idêntica —
ele é normalizado pela demanda da cultura ao longo do ciclo específico, não pelo déficit
anual total.

1. Para cada município e janela do exercício acima, calcule também o `DEF` total do ciclo
   (déficit hídrico em mm, disponível no DataFrame retornado por `calcular_isna`) e compare
   com o ISNA. Eles concordam sobre qual é a melhor janela?
2. Discuta: em que situação um método (ISNA normalizado) e outro (Dha absoluto em mm) podem
   discordar sobre qual município ou janela é menos arriscado?
3. *(opcional)* Substitua os dados ilustrativos por dados reais de precipitação e ETo (Capítulos
   1 e 5) para 2-3 municípios do Paraná de seu interesse, agregados mensalmente.


In [ ]:
# Espaço para o desafio — escreva seu código aqui


## 7.6 Checkpoint

Antes de seguir para o **Capítulo 8 — Predição de produtividade com Machine Learning**,
você deve ter:

- [ ] uma função `calcular_isna(P, ETc, CAD)` funcionando, reaproveitando
      `amp.balanco_hidrico_cultura` da biblioteca;
- [ ] uma tabela de ISNA por município e janela de semeadura, com a melhor janela identificada
      para cada município;
- [ ] resolvido pelo menos o item 1 do desafio.

O `ISNA` calculado aqui, junto com `Ta` (temperatura média) e `Dha` (déficit hídrico), vira
**variável preditora** no Capítulo 8 — a lógica determinística (regras de corte) dá lugar a
um modelo estatístico que aprende a relação entre essas variáveis e a produtividade.